# Context-aware fraud decision system

**Goal:** Move beyond raw fraud probability to a **policy** that balances fraud loss, false blocks, and OTP friction.

**Dataset:** ULB credit card fraud (`data/creditcard.csv`) — PCA features (`V1`–`V28`), `Time`, `Amount`, `Class`.

**Design choices (read first):**
- **Time-based split** (no shuffle) to mimic production scoring on future transactions.
- **Pseudo users** (`sim_user_id`): MiniBatchKMeans on PCA space — *not* real cardholders; stand-in for velocity/trust stories.
- **Auth / device signals** (`otp_*`, `consistent_device_flag`, etc.) are **simulated** from transaction rhythm because the CSV has no auth logs. Comments mark every assumption.
- **Intent / confusion** layers are **heuristic scores** you would replace with real login/OTP telemetry in production.
- **§11–12** add **review-queue capacity**, **streaming dynamic trust**, **regret**, **segments**, **UAE-style scenario labels**, and a **Key findings** rollup — without changing the fraud model.


In [ ]:
# --- 0) Imports & paths ---
from __future__ import annotations

import json
import warnings
from dataclasses import dataclass
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.cluster import MiniBatchKMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
RNG = np.random.default_rng(RANDOM_STATE)

# Resolve project root: works if kernel cwd is repo root or notebooks/
_here = Path.cwd().resolve()
if (_here / "data" / "creditcard.csv").exists():
    PROJECT_ROOT = _here
elif (_here.parent / "data" / "creditcard.csv").exists():
    PROJECT_ROOT = _here.parent
else:
    raise FileNotFoundError("Place creditcard.csv under <repo>/data/creditcard.csv")

DATA_PATH = PROJECT_ROOT / "data" / "creditcard.csv"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

V_COLS = [f"V{i}" for i in range(1, 29)]


## 1) Data loading


In [ ]:
df = pd.read_csv(DATA_PATH)
assert {"Time", "Amount", "Class"}.issubset(df.columns)
df["Class"] = df["Class"].astype(int)
print(df.shape, "fraud rate:", f"{df['Class'].mean():.4f}")


## 2) Time-based split (chronological)


In [ ]:
df = df.sort_values("Time", kind="mergesort").reset_index(drop=True)

n = len(df)
train_end = int(0.70 * n)
val_end = int(0.85 * n)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

y_train = train_df["Class"].astype(int)
y_val = val_df["Class"].astype(int)
y_test = test_df["Class"].astype(int)

print("Train", train_df.shape, "fraud rate", f"{y_train.mean():.5f}")
print("Val  ", val_df.shape, "fraud rate", f"{y_val.mean():.5f}")
print("Test ", test_df.shape, "fraud rate", f"{y_test.mean():.5f}")


## 3) Feature engineering (pseudo-users, velocity, simulated auth)


In [ ]:
# --- Pseudo users (MiniBatchKMeans on PCA features only; k is a design knob) ---
N_USER_CLUSTERS = 1200
_kmeans = MiniBatchKMeans(
    n_clusters=N_USER_CLUSTERS,
    random_state=RANDOM_STATE,
    batch_size=8192,
    n_init="auto",
)
_kmeans.fit(train_df[V_COLS])  # fit only on train to avoid test leakage for cluster assignment
# Assign all splits with the same centroids (production: assign online with frozen model)
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    part["sim_user_id"] = _kmeans.predict(part[V_COLS])


def _velocity_last_window(times: np.ndarray, window_sec: float) -> np.ndarray:
    # Per-user times must be sorted ascending.
    n = len(times)
    out = np.ones(n, dtype=np.float64)
    j = 0
    for i in range(n):
        while times[i] - times[j] > window_sec:
            j += 1
        out[i] = i - j + 1
    return out


def add_behavioral_features(part: pd.DataFrame, window_sec: float = 3600.0) -> pd.DataFrame:
    # Per pseudo-user features in chronological order within that user.
    # Assumption: global Time is comparable within dataset (seconds since first txn in source data).
    d = part.sort_values(["sim_user_id", "Time"], kind="mergesort").copy()
    d["time_since_last_transaction"] = d.groupby("sim_user_id")["Time"].diff()
    d["time_since_last_transaction"] = d["time_since_last_transaction"].fillna(np.nanmedian(d["time_since_last_transaction"]))

    d["transaction_velocity"] = d.groupby("sim_user_id", group_keys=False)["Time"].transform(
        lambda t: _velocity_last_window(t.values.astype(np.float64), window_sec)
    )

    d["rolling_avg_amount"] = d.groupby("sim_user_id")["Amount"].transform(
        lambda s: s.rolling(window=5, min_periods=1).mean()
    )
    mu = d.groupby("sim_user_id")["Amount"].transform(lambda s: s.expanding().mean().shift(1))
    d["deviation_from_user_mean"] = d["Amount"] - mu.fillna(d["Amount"])

    d["user_txn_index"] = d.groupby("sim_user_id").cumcount()
    max_idx = d.groupby("sim_user_id")["user_txn_index"].transform("max")
    d["trust_score_base"] = (d["user_txn_index"] / (max_idx + 1)).clip(0, 1)

    # Travel / context jump: long gap since last txn for this pseudo-user (tunable)
    gap_thr = d["time_since_last_transaction"].quantile(0.92)
    d["simulated_travel_flag"] = ((d["time_since_last_transaction"] > gap_thr) & (d["transaction_velocity"] <= 2)).astype(int)

    # --- Simulated auth-channel signals (NOT in CSV; stand-ins for real OTP/device logs) ---
    # Higher expected retries when user slams many txns quickly (confusion or panic).
    lam_retry = 0.08 + 0.12 * np.tanh(d["transaction_velocity"] / 8) + 0.06 * np.tanh(120 / (d["time_since_last_transaction"] + 1))
    d["otp_retry_count"] = RNG.poisson(np.clip(lam_retry, 0.02, 1.5))
    d["otp_resend_count"] = RNG.poisson(np.clip(0.05 + 0.2 * (d["otp_retry_count"] > 0), 0.02, 1.2))

    d["time_between_attempts"] = d.groupby("sim_user_id")["Time"].diff().fillna(10_000.0)
    # Device sticky for pseudo-user with rare flips (simulated churn / new device)
    p_flip = 0.05 + 0.15 * d["simulated_travel_flag"]
    d["consistent_device_flag"] = (RNG.random(len(d)) > p_flip).astype(np.int8)

    # --- Confusion vs attack heuristics ---
    is_fast = (d["time_between_attempts"] < 120).astype(np.float32)
    device = d["consistent_device_flag"].astype(np.float32)
    retry_n = np.clip(d["otp_retry_count"] / 4.0, 0, 1)
    resend_n = np.clip(d["otp_resend_count"] / 3.0, 0, 1)
    # Many retries + stable device + not ultra-fast typing → human confusion
    human_confusion = retry_n * device * (1 - is_fast)
    d["confusion_score"] = np.clip(0.55 * human_confusion + 0.25 * resend_n + 0.15 * retry_n, 0, 1)

    # Fast + flaky device + resends → bot / ATO-style pressure
    attack_pressure = is_fast * (1 - device) * (0.4 + 0.6 * resend_n)
    calm = (1 - is_fast) * (0.2 + 0.8 * device) * (1 - retry_n * 0.5)
    d["intent_numeric"] = np.clip(calm - attack_pressure, -1.5, 1.5)

    def _intent_label(x: float) -> str:
        if x > 0.35:
            return "likely_human"
        if x < -0.35:
            return "likely_attack"
        return "uncertain"

    d["intent_label"] = d["intent_numeric"].map(_intent_label)

    amt_std = d.groupby("sim_user_id")["Amount"].transform(lambda s: s.expanding().std().shift(1))
    amt_std = amt_std.fillna(d["Amount"].std() + 1e-6)
    amt_z = (d["Amount"] - mu.fillna(d.groupby("sim_user_id")["Amount"].transform("mean"))) / (amt_std + 1e-6)
    d["trust_score"] = (
        0.55 * d["trust_score_base"] + 0.25 * (1 - np.clip(np.abs(amt_z) / 3.0, 0, 1)) + 0.20 * (1 - d["simulated_travel_flag"])
    ).clip(0, 1)

    # Restore original row order within this split
    d = d.sort_index()
    return d


train_df = add_behavioral_features(train_df)
val_df = add_behavioral_features(val_df)
test_df = add_behavioral_features(test_df)

# intent_label → one-hot; align columns across splits (val/test may miss a rare label)
INTENT_OHE = sorted(pd.get_dummies(train_df["intent_label"], prefix="intent").columns.tolist())
for part in (train_df, val_df, test_df):
    d = pd.get_dummies(part["intent_label"], prefix="intent")
    part[INTENT_OHE] = d.reindex(columns=INTENT_OHE, fill_value=0)
MODEL_FEATURES = [c for c in train_df.columns if c not in ("Class", "intent_label")]

X_train = train_df[MODEL_FEATURES]
X_val = val_df[MODEL_FEATURES]
X_test = test_df[MODEL_FEATURES]

print("Model feature count:", len(MODEL_FEATURES))


## 4) Model training (LR + XGB with class imbalance)


In [ ]:
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / max(pos, 1)

lr_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=4000, random_state=RANDOM_STATE)),
    ]
)
lr_model.fit(X_train, y_train)

xgb_core = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
)
xgb_core.fit(X_train, y_train)

val_prob_lr = lr_model.predict_proba(X_val)[:, 1]
val_prob_xgb = xgb_core.predict_proba(X_val)[:, 1]
print("Val AP  LR :", average_precision_score(y_val, val_prob_lr))
print("Val AP XGB:", average_precision_score(y_val, val_prob_xgb))


## 5) Calibration (probabilities for policy, not just ranking)


In [ ]:
tscv = TimeSeriesSplit(n_splits=3)

calibrated = CalibratedClassifierCV(
    estimator=XGBClassifier(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
    ),
    method="sigmoid",
    cv=tscv,
)
calibrated.fit(X_train, y_train)

val_prob = calibrated.predict_proba(X_val)[:, 1]
test_prob = calibrated.predict_proba(X_test)[:, 1]
print("Val AP (calibrated):", average_precision_score(y_val, val_prob))
print("Test AP (calibrated):", average_precision_score(y_test, test_prob))


## 6) Decision system (combine fraud + friction + trust)

**Inputs:** calibrated `fraud_probability`, `confusion_score`, `intent_label`, `trust_score`, `Amount`.

**Priority (vectorized in code):**
1. **BLOCK** if effective risk `p_eff` ≥ `t_block`, OR (`intent == likely_attack` AND `p_eff` ≥ `attack_block`).
2. **OTP** if `t_review` ≤ `p_eff` < `t_block` AND confusion is high (step-up instead of hard decline), OR remaining medium-risk traffic.
3. **APPROVE** if fraud is low and trust is high, OR `p_eff` is very low.

`p_eff` adds a small amount-aware nudge (higher `Amount` → slightly more conservative).


In [ ]:
@dataclass
class PolicyThresholds:
    t_block: float = 0.82
    t_review: float = 0.28
    confusion_otp: float = 0.42
    trust_soft: float = 0.58
    attack_block: float = 0.18  # min fraud p to block when intent says attack


def apply_policy_vec(df: pd.DataFrame, thr: PolicyThresholds, trust_col: str = "trust_score") -> np.ndarray:
    # Vectorized policy: BLOCK / OTP / APPROVE (see markdown section above for readable rules).
    # trust_col lets streaming simulation swap in dynamic trust without retraining the model.
    p = df["fraud_probability"].to_numpy(dtype=np.float64)
    c = df["confusion_score"].to_numpy(dtype=np.float64)
    t = df[trust_col].to_numpy(dtype=np.float64)
    intent = df["intent_label"].astype(str).to_numpy()
    amt = df["Amount"].to_numpy(dtype=np.float64)

    p_eff = p + 0.03 * np.tanh(amt / 200.0)

    dec = np.full(len(df), "APPROVE", dtype=object)
    blocked = (p_eff >= thr.t_block) | ((intent == "likely_attack") & (p_eff >= thr.attack_block))
    dec[blocked] = "BLOCK"

    active = ~blocked
    otp1 = active & (p_eff >= thr.t_review) & (p_eff < thr.t_block) & (c >= thr.confusion_otp)
    dec[otp1] = "OTP"
    active = active & ~otp1

    app1 = active & (p_eff < thr.t_review) & (t >= thr.trust_soft)
    dec[app1] = "APPROVE"
    active = active & ~app1

    app2 = active & (p_eff < thr.t_review * 0.65)
    dec[app2] = "APPROVE"
    active = active & ~app2

    otp2 = active & (p_eff >= thr.t_review)
    dec[otp2] = "OTP"
    return dec


def attach_scores(df: pd.DataFrame, prob: np.ndarray) -> pd.DataFrame:
    out = df.copy()
    out["fraud_probability"] = prob
    return out


val_scored = attach_scores(val_df, val_prob)
test_scored = attach_scores(test_df, test_prob)


## 7) Friction-aware cost model + threshold search (validation only)


In [ ]:
# Dollar-shaped units — tune to your business (relative ratios matter most).
COST_FN = 200.0       # missed fraud
COST_BLOCK_FP = 25.0  # blocking a good customer
COST_OTP_LEGIT = 6.0  # OTP friction on legitimate user
# Fraud that hits OTP still creates partial loss (not fully stopped)
COST_FN_OTP = 70.0


def total_cost(y_true: np.ndarray, decisions: list[str]) -> float:
    cost = 0.0
    for yt, d in zip(y_true, decisions):
        is_fraud = yt == 1
        if is_fraud:
            if d == "APPROVE":
                cost += COST_FN
            elif d == "OTP":
                cost += COST_FN_OTP
            # BLOCK → contained
        else:
            if d == "BLOCK":
                cost += COST_BLOCK_FP
            elif d == "OTP":
                cost += COST_OTP_LEGIT
    return cost


def eval_policy(df_scored: pd.DataFrame, y: np.ndarray, thr: PolicyThresholds, trust_col: str = "trust_score"):
    decs = apply_policy_vec(df_scored, thr, trust_col=trust_col).tolist()
    return decs, total_cost(y, decs)


def decision_rates(decisions: list[str]) -> dict[str, float]:
    s = pd.Series(decisions)
    return (s.value_counts(normalize=True)).reindex(["APPROVE", "OTP", "BLOCK"]).fillna(0).to_dict()


# --- Grid search on validation (coarse grid; vectorized policy keeps this fast) ---
best = None
for t_block in np.linspace(0.55, 0.92, 6):
    for t_rev in np.linspace(0.12, 0.42, 5):
        for c_otp in np.linspace(0.25, 0.60, 4):
            for t_tr in np.linspace(0.45, 0.72, 3):
                thr = PolicyThresholds(
                    t_block=float(t_block),
                    t_review=float(t_rev),
                    confusion_otp=float(c_otp),
                    trust_soft=float(t_tr),
                )
                decs, c = eval_policy(val_scored, y_val.values, thr)
                if best is None or c < best[0]:
                    best = (c, thr, decs)

best_cost_val, best_thr, _ = best
print("Best validation total cost:", best_cost_val)
print(best_thr)

policy_thr = best_thr


## 8) Evaluation: precision@k, recall@k, costs, decision mix


In [ ]:
def precision_recall_at_k(y_true: np.ndarray, y_score: np.ndarray, ks=(50, 100, 250, 500, 1000)):
    order = np.argsort(-y_score)
    y_sorted = y_true[order]
    out = {}
    n_pos = max(int(y_true.sum()), 1)
    for k in ks:
        k = min(k, len(y_true))
        top = y_sorted[:k]
        out[f"P@{k}"] = top.mean()
        out[f"R@{k}"] = top.sum() / n_pos
    return out


prk = precision_recall_at_k(y_test.values, test_prob)
print("Test ranking metrics:", json.dumps(prk, indent=2))


def naive_threshold_decisions(p: np.ndarray, t: float = 0.5):
    return ["BLOCK" if x >= t else "APPROVE" for x in p]


# Tune single threshold on val by cost (same cost function, no OTP path distinction beyond block/approve)
def best_single_threshold(y_val_arr: np.ndarray, p_val: np.ndarray) -> tuple[float, float]:
    best_t, best_c = 0.5, float("inf")
    for t in np.linspace(0.02, 0.98, 49):
        decs = naive_threshold_decisions(p_val, t)
        c = total_cost(y_val_arr, decs)
        if c < best_c:
            best_c, best_t = c, t
    return best_t, best_c


t_naive, naive_val_cost = best_single_threshold(y_val.values, val_prob)
naive_test_dec = naive_threshold_decisions(test_prob, t_naive)
naive_test_cost = total_cost(y_test.values, naive_test_dec)

policy_test_dec, policy_test_cost = eval_policy(test_scored, y_test.values, policy_thr)

print(f"Naive threshold (val-tuned) t={t_naive:.3f} | test cost = {naive_test_cost:,.0f}")
print(f"Policy system          | test cost = {policy_test_cost:,.0f}")
if naive_test_cost > 0:
    print(f"Cost vs naive: {(1 - policy_test_cost / naive_test_cost) * 100:+.1f}%")

print("Decision mix (policy, test):", decision_rates(policy_test_dec))
print("Decision mix (naive, test):", decision_rates(naive_test_dec))


## 9) Story-driven diagnostics (OTP path, confused users)


In [ ]:
test_scored = test_scored.copy()
test_scored["policy_decision"] = policy_test_dec
test_scored["naive_decision"] = naive_test_dec
test_scored["y_true"] = y_test.values

otp_mask = np.array(policy_test_dec) == "OTP"
if otp_mask.any():
    otp_df = test_scored.loc[otp_mask]
    legit_share = 1 - otp_df["y_true"].mean()
    fraud_share = otp_df["y_true"].mean()
    high_confusion = (otp_df["confusion_score"] >= 0.35).mean()
    print(
        f"OTP path: {100 * legit_share:.1f}% were legitimate (confused-user / step-up candidates), "
        f"{100 * fraud_share:.1f}% fraud."
    )
    print(f"Among OTP, {100 * high_confusion:.1f}% had elevated confusion_score (>=0.35).")

# Unnecessary OTP: legit users forced through OTP under policy
legit_otp = test_scored[(test_scored["y_true"] == 0) & (test_scored["policy_decision"] == "OTP")]
naive_legit_blocked = test_scored[(test_scored["y_true"] == 0) & (test_scored["naive_decision"] == "BLOCK")]
policy_legit_blocked = test_scored[(test_scored["y_true"] == 0) & (test_scored["policy_decision"] == "BLOCK")]
print("Legit OTP count (policy):", len(legit_otp))
print("Legit blocked count — naive vs policy:", len(naive_legit_blocked), "→", len(policy_legit_blocked))

otp_legit = len(legit_otp)
naive_fp_blocks = len(naive_legit_blocked)
if naive_fp_blocks > 0 and otp_legit <= naive_fp_blocks:
    # proxy: friction shifted from hard blocks to OTP
    print(
        f"Rough friction shift: policy issues {otp_legit} OTPs to legit users vs {naive_fp_blocks} hard blocks under naive threshold "
        f"(not apples-to-apples, but shows security/friction trade-space)."
    )

# Plot PR curve
p, r, _ = precision_recall_curve(y_test, test_prob)
plt.figure(figsize=(6, 4))
plt.plot(r, p, label=f"PR (AP={average_precision_score(y_test, test_prob):.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Test precision–recall (calibrated fraud model)")
plt.legend()
plt.tight_layout()
plt.show()


## 11) Production-style simulation layers

Operational constraints on the **frozen** model: review capacity, streaming + dynamic trust, regret, segments, UAE-style labels. `fraud_probability` is **not** recomputed.


### 11a) Review queue (finite analyst capacity)

Synthetic days = equal-width `Time` slices. Each day only top `DAILY_REVIEW_CAPACITY` by `fraud_probability` are reviewed; others **auto-APPROVE** (tail fraud missed). Reviewed rows keep static policy; overflow uses APPROVE.


In [ ]:
DAILY_REVIEW_CAPACITY = 500
N_SYNTH_DAYS = 14  # equal-width time buckets over the test window

_ord = test_scored.sort_values("Time")
static_policy_ser = pd.Series(policy_test_dec, index=test_scored.index)
dec_static_ordered = static_policy_ser.loc[_ord.index].values
y_ord = _ord["y_true"].values
p_ord = _ord["fraud_probability"].values
tmin, tmax = _ord["Time"].min(), _ord["Time"].max()
span = max(tmax - tmin, 1e-6)
day_bucket = ((_ord["Time"] - tmin) / span * N_SYNTH_DAYS).astype(int).clip(0, N_SYNTH_DAYS - 1).values

dec_queue = np.full(len(_ord), "APPROVE", dtype=object)
reviewed_mask = np.zeros(len(_ord), dtype=bool)
daily_prk = []

for d in range(N_SYNTH_DAYS):
    idx = np.where(day_bucket == d)[0]
    if len(idx) == 0:
        continue
    sub_p = p_ord[idx]
    sub_y = y_ord[idx]
    k = min(DAILY_REVIEW_CAPACITY, len(idx))
    top_local = np.argsort(-sub_p)[:k]
    top_global = idx[top_local]
    reviewed_mask[top_global] = True
    dec_queue[top_global] = dec_static_ordered[top_global]
    top_y = sub_y[top_local]
    daily_prk.append(
        {"day": d, "P@K": float(top_y.mean()), "R@K": float(top_y.sum() / max(int(sub_y.sum()), 1)), "K": k, "n_txn": len(idx)}
    )

cost_all_reviewed = total_cost(y_ord, dec_static_ordered.tolist())
cost_queue = total_cost(y_ord, dec_queue.tolist())
missed_fraud_queue = int(((y_ord == 1) & (~reviewed_mask)).sum())
fraud_in_test = int((y_ord == 1).sum())

print(f"Daily review capacity: {DAILY_REVIEW_CAPACITY} | synthetic days: {N_SYNTH_DAYS}")
print(f"Cost if every txn reviewed (static policy): {cost_all_reviewed:,.0f}")
print(f"Cost with queue + auto-approve overflow: {cost_queue:,.0f}")
if cost_all_reviewed > 0:
    print(f"Queue vs full-review cost delta: {(cost_queue - cost_all_reviewed) / cost_all_reviewed * 100:+.1f}%")
print(f"Fraud never reaching review (overflow tail): {missed_fraud_queue} / {fraud_in_test}")
if daily_prk:
    mean_p = float(np.mean([x["P@K"] for x in daily_prk]))
    mean_r = float(np.mean([x["R@K"] for x in daily_prk]))
    print(f"Mean daily P@K (top-{DAILY_REVIEW_CAPACITY} within each bucket): {mean_p:.4f}")
    print(f"Mean daily R@K (fraud caught in top-K vs fraud in bucket): {mean_r:.4f}")
print("Global test P@K / R@K (section 8, full-test ranking):", json.dumps(prk, indent=2))
print("First 3 daily buckets:", json.dumps(daily_prk[:3], indent=2))


### 11b) UAE-style context labels

**Travelers:** random pseudo-users `uae_frequent_traveler` + optional `uae_travel_spike` (analysis only). **Shared accounts:** high train `Amount` CV → `shared_account_suspect`.


In [ ]:
uids = test_scored["sim_user_id"].unique()
n_pick = max(1, int(0.08 * len(uids)))
uae_user_set = set(RNG.choice(uids, size=n_pick, replace=False))
test_scored = test_scored.copy()
test_scored["uae_frequent_traveler"] = test_scored["sim_user_id"].isin(uae_user_set).astype(np.int8)
test_scored["uae_travel_spike"] = (
    (test_scored["uae_frequent_traveler"] == 1) & (RNG.random(len(test_scored)) < 0.15)
).astype(np.int8)
train_amt_cv = train_df.groupby("sim_user_id")["Amount"].apply(lambda s: float(s.std() / (s.mean() + 1e-6)))
cv_thr = train_amt_cv.quantile(0.85)
shared_set = set(train_amt_cv[train_amt_cv > cv_thr].index)
test_scored["shared_account_suspect"] = test_scored["sim_user_id"].isin(shared_set).astype(np.int8)
print("UAE pseudo-users:", len(uae_user_set), "| shared-account suspect rows:", int(test_scored["shared_account_suspect"].sum()))


### 11c) Streaming timeline + dynamic user trust

Walk test rows in `Time` order. Oracle `y_true` updates trust after each step (zero lag — production would lag).


In [ ]:
_stream = test_scored.sort_values("Time")
uids_arr = _stream["sim_user_id"].to_numpy()
y_arr = _stream["y_true"].to_numpy()
trust0 = _stream["trust_score"].to_numpy(dtype=np.float64)
p_arr = _stream["fraud_probability"].to_numpy(dtype=np.float64)
c_arr = _stream["confusion_score"].to_numpy(dtype=np.float64)
amt_arr = _stream["Amount"].to_numpy(dtype=np.float64)
intent_arr = _stream["intent_label"].astype(str).to_numpy()
dev_arr = _stream["deviation_from_user_mean"].to_numpy(dtype=np.float64)
dev_thresh = float(np.percentile(np.abs(dev_arr), 60))

n = len(_stream)
trust_dyn = np.zeros(n, dtype=np.float64)
dec_stream_list = []
user_trust, user_fraud_flags, user_stability, user_txn = {}, {}, {}, {}

for i in range(n):
    uid = int(uids_arr[i])
    if uid not in user_trust:
        user_trust[uid] = float(trust0[i])
        user_fraud_flags[uid] = 0
        user_stability[uid] = 1.0
        user_txn[uid] = 0
    td = user_trust[uid]
    trust_dyn[i] = td
    row = pd.DataFrame(
        {
            "fraud_probability": [p_arr[i]],
            "confusion_score": [c_arr[i]],
            "trust_dynamic": [td],
            "intent_label": [intent_arr[i]],
            "Amount": [amt_arr[i]],
        }
    )
    dec = apply_policy_vec(row, policy_thr, trust_col="trust_dynamic")[0]
    dec_stream_list.append(dec)
    yt = int(y_arr[i])
    if dec in ("BLOCK", "OTP") and p_arr[i] >= policy_thr.t_review:
        user_fraud_flags[uid] += 1
    if yt == 1:
        user_trust[uid] = max(0.0, user_trust[uid] * 0.55)
    else:
        if dec == "APPROVE":
            user_trust[uid] = min(1.0, user_trust[uid] + 0.018)
        elif dec == "OTP":
            user_trust[uid] = min(1.0, user_trust[uid] + 0.010)
        else:
            user_trust[uid] = max(0.0, user_trust[uid] - 0.045)
    stab_signal = 1.0 if abs(dev_arr[i]) < dev_thresh else 0.0
    user_stability[uid] = 0.88 * user_stability[uid] + 0.12 * stab_signal
    user_txn[uid] += 1

dec_stream = np.array(dec_stream_list, dtype=object)
_stream = _stream.copy()
_stream["trust_dynamic"] = trust_dyn
_stream["policy_streaming"] = dec_stream
cost_stream = total_cost(y_arr, dec_stream.tolist())
print("Streaming (dynamic trust) test cost:", f"{cost_stream:,.0f}")
print("Static policy test cost (reference):", f"{policy_test_cost:,.0f}")
print("Pseudo-users with >=1 review-ish flag:", sum(1 for v in user_fraud_flags.values() if v > 0))


### 11d) Regret decomposition


In [ ]:
legit = y_arr == 0
fraud = y_arr == 1
regret_block_legit = float((legit & (dec_stream == "BLOCK")).sum() * COST_BLOCK_FP)
regret_allow_fraud = float((fraud & (dec_stream == "APPROVE")).sum() * COST_FN)
regret_otp_friction = float((legit & (dec_stream == "OTP")).sum() * COST_OTP_LEGIT)
regret_fraud_otp = float((fraud & (dec_stream == "OTP")).sum() * COST_FN_OTP)
print("regret_block_legit:", f"${regret_block_legit:,.0f}", "| regret_allow_fraud:", f"${regret_allow_fraud:,.0f}")
print("regret_otp_friction (legit):", f"${regret_otp_friction:,.0f}", "| fraud_on_otp_path:", f"${regret_fraud_otp:,.0f}")


### 11e) Segmented analysis (Amount x dynamic trust)


In [ ]:
amt_med = _stream["Amount"].median()
trust_med = float(np.median(trust_dyn))
_stream["seg_amount"] = np.where(_stream["Amount"] >= amt_med, "high_amt", "low_amt")
_stream["seg_trust"] = np.where(trust_dyn >= trust_med, "high_trust_dyn", "low_trust_dyn")

def seg_summary(sub, name):
    if len(sub) == 0:
        return
    d = sub["policy_streaming"]
    print(name, "n=", len(sub), "fraud_rate=", f"{sub['y_true'].mean():.4f}", "block%=", f"{(d == 'BLOCK').mean():.3f}", "otp%=", f"{(d == 'OTP').mean():.3f}")

for a in ["low_amt", "high_amt"]:
    for t in ["low_trust_dyn", "high_trust_dyn"]:
        mask = (_stream["seg_amount"] == a) & (_stream["seg_trust"] == t)
        seg_summary(_stream[mask], f"{a} + {t}")
for label, col in [("UAE traveler", "uae_frequent_traveler"), ("Travel spike", "uae_travel_spike"), ("Shared acct suspect", "shared_account_suspect")]:
    seg_summary(_stream[_stream[col] == 1], label)


## 12) Key findings (auto summary)


In [ ]:
otp_mask_sf = dec_stream == "OTP"
otp_legit_pct = (
    float((y_arr[otp_mask_sf] == 0).mean() * 100) if otp_mask_sf.any() else 0.0
)
naive_series = pd.Series(naive_test_dec, index=test_scored.index)
naive_on_stream = naive_series.loc[_stream.index].values
policy_static_on_stream = static_policy_ser.loc[_stream.index].values
naive_blocks_legit = int(((y_arr == 0) & (naive_on_stream == "BLOCK")).sum())
stream_blocks_legit = int(((y_arr == 0) & (dec_stream == "BLOCK")).sum())
policy_static_blocks_legit = int(((y_arr == 0) & (policy_static_on_stream == "BLOCK")).sum())
pct_block_reduction_vs_naive = (1 - stream_blocks_legit / max(naive_blocks_legit, 1)) * 100
pct_block_reduction_vs_static_policy = (1 - stream_blocks_legit / max(policy_static_blocks_legit, 1)) * 100
queue_cost_delta_pct = (
    (cost_queue - cost_all_reviewed) / cost_all_reviewed * 100 if cost_all_reviewed > 0 else 0.0
)
missed_fraud_pct = missed_fraud_queue / max(fraud_in_test, 1) * 100

KEY_FINDINGS = {
    "otp_path_legit_share_pct": round(otp_legit_pct, 2),
    "legit_blocks_naive_vs_streaming": [naive_blocks_legit, stream_blocks_legit],
    "legit_blocks_static_policy_vs_streaming": [policy_static_blocks_legit, stream_blocks_legit],
    "pct_fewer_legit_blocks_streaming_vs_naive": round(pct_block_reduction_vs_naive, 2),
    "pct_fewer_legit_blocks_streaming_vs_static_batch": round(pct_block_reduction_vs_static_policy, 2),
    "queue_cost_increase_pct_vs_full_review": round(queue_cost_delta_pct, 2),
    "pct_of_test_fraud_missed_due_to_queue_overflow": round(missed_fraud_pct, 2),
    "regret_usd": {
        "block_legit": regret_block_legit,
        "allow_fraud": regret_allow_fraud,
        "otp_friction_legit": regret_otp_friction,
        "fraud_on_otp_path": regret_fraud_otp,
    },
}

print("=== Key findings ===")
print(
    f"• {KEY_FINDINGS['otp_path_legit_share_pct']:.1f}% of streaming OTP decisions hit legitimate users (confused / step-up), not fraud."
)
print(
    f"• Legit blocks: naive threshold {naive_blocks_legit} → streaming dynamic-trust {stream_blocks_legit} "
    f"(~{KEY_FINDINGS['pct_fewer_legit_blocks_streaming_vs_naive']:.1f}% vs naive on same timeline)."
)
print(
    f"• Legit blocks: batch policy {policy_static_blocks_legit} → streaming {stream_blocks_legit} "
    f"(~{KEY_FINDINGS['pct_fewer_legit_blocks_streaming_vs_static_batch']:.1f}% change)."
)
print(
    f"• Queue cap moves total cost by ~{KEY_FINDINGS['queue_cost_increase_pct_vs_full_review']:.1f}% vs reviewing everyone; "
    f"~{KEY_FINDINGS['pct_of_test_fraud_missed_due_to_queue_overflow']:.1f}% of period fraud never made the daily top-{DAILY_REVIEW_CAPACITY} review slots."
)
print(
    f"• Regret: block-legit ${regret_block_legit:,.0f}, allow-fraud ${regret_allow_fraud:,.0f}, OTP friction ${regret_otp_friction:,.0f}."
)


## 10) Persist artifacts


In [ ]:
bundle = {
    "calibrated_model": calibrated,
    "feature_columns": MODEL_FEATURES,
    "kmeans_users": _kmeans,
    "policy_thresholds": policy_thr,
    "costs": {
        "COST_FN": COST_FN,
        "COST_BLOCK_FP": COST_BLOCK_FP,
        "COST_OTP_LEGIT": COST_OTP_LEGIT,
        "COST_FN_OTP": COST_FN_OTP,
    },
    "naive_threshold_val_tuned": float(t_naive),
    "review_queue": {"daily_capacity": DAILY_REVIEW_CAPACITY, "synth_days": N_SYNTH_DAYS},
    "key_findings": KEY_FINDINGS,
}

model_path = MODEL_DIR / "xgb_calibrated_fraud_decision.joblib"
meta_path = MODEL_DIR / "decision_system_meta.json"

joblib.dump(bundle, model_path)
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "policy": policy_thr.__dict__,
            "feature_count": len(MODEL_FEATURES),
            "model_path": str(model_path.name),
        },
        f,
        indent=2,
    )

print("Saved:", model_path)
print("Meta:", meta_path)
